<a href="https://colab.research.google.com/github/mugalan/classical-mechanics-from-a-geometric-point-of-view/blob/main/mechanics/answers-to-selected-assignments/Inertial_Observers_Assignment_Sample_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Inertial Observers


* These assignments supplement the note:

https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view/blob/main/mechanics/class-notes/Mugas_Classical_Mechanics.ipynb

# Setting Up

In [ ]:
import numpy as np
import scipy as sp
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy

from sympy import symbols
from sympy import *

import plotly.graph_objects as go

In [ ]:
# Install the simulation helpers
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

#Q1


Plot an orthonormal frame. You may use the following or a similar package

https://plotly.com/python/

In [ ]:
#Adding a Frame
fig = go.Figure()
fig=mr.add_orth_norm_frame(fig,np.array([0,0,0]),np.array([[1,0,0],[0,1,0],[0,0,1]]),[[-2,2],[-2,2],[-2,2]],'blue')
fig.show()

# Q2


 A point $P$ in space has the Euclidean representations $P\equiv (1,1,1)$ with respect to some inertial frame $\mathbf{e}$ with origin coinciding with a point $O$ in space. Find the angle that $OP$ makes with the plane spanned by the $\mathbf{e}_1$, $\mathbf{e}_2$ directions. Verify your answer and plot an orthonormal frame and the point $P$ in that frame.

In [ ]:
P=[1,1,1]; Q=[1,1,0]; O=[0,0,0];
PQ=np.array(P)-np.array(Q)
lengthPQ=linalg.norm(PQ)
anglePQ=np.arccos(np.array(P).dot(np.array(Q))/(linalg.norm(P)*linalg.norm(Q)))
print(['length='+str(lengthPQ), 'angle='+str(180*anglePQ/np.pi)])

In [ ]:
#Adding a Frame
fig = go.Figure()
fig=mr.add_orth_norm_frame(fig,O,np.array([[1,0,0],[0,1,0],[0,0,1]]),[[-2,2],[-2,2],[-2,2]],'blue')
fig.show()

In [ ]:
#Adding the Points
fig.add_trace(go.Scatter3d(x=[O[0],P[0]], y=[O[1],P[1]], z=[O[2],P[2]], hoverinfo='x+y+z', mode='lines', line=dict(width=8)))
fig.add_trace(go.Scatter3d(x=[O[0],Q[0]], y=[O[1],Q[1]], z=[O[2],Q[2]], hoverinfo='x+y+z', mode='lines', line=dict(width=8)))

# Q3

Consider an orthonormal frame $\mathbf e$ with
origin $O$. Let $P,Q$ be two points such that $O,P,Q$
are non-collinear, with representations $x_P,x_Q$.
Let $R:[0,\infty)\to SO(3)$, and let $P_R,Q_R$ have
representations $R(t)x_P,R(t)x_Q$.
When the planes through $O,P,Q$ and $O,P_R,Q_R$ are
distinct, find an explicit expression for their line
of intersection. Identify the condition under which
the planes coincide.

## Sample Answer

Since $O,P,Q$ are non-collinear,
$$
n=x_P\times x_Q\ne0.
$$
The plane through $O,P,Q$ has equation
$$
n^Tx=0.
$$
A normal to the plane through $O,P_R,Q_R$ is
$$
n_R(t)
=\bigl(R(t)x_P\bigr)\times\bigl(R(t)x_Q\bigr)
=R(t)n,
$$
so its equation is
$$
\bigl(R(t)n\bigr)^Tx=0.
$$

When the planes are distinct, their normals are
not parallel. Hence
$$
d(t)=n\times R(t)n\ne0.
$$
This direction is perpendicular to both normals:
$$
n^Td(t)=0,\qquad
\bigl(R(t)n\bigr)^Td(t)=0.
$$
Both planes pass through $O$, so their line of
intersection is
$$
\boxed{
\mathscr L(t)=
\left\{x=s\,\frac{n\times R(t)n}
{\|n\times R(t)n\|}:s\in\mathbb R\right\},
\qquad n=x_P\times x_Q.
}
$$
Indeed, the two independent plane equations impose
two independent linear conditions on $x\in\mathbb R^3$,
leaving precisely this one-dimensional set.

The planes coincide exactly when their normals
are parallel:
$$
n\times R(t)n=0.
$$
Since $R(t)$ preserves length, this is equivalent to
$$
\boxed{R(t)n=n\quad\text{or}\quad R(t)n=-n.}
$$
In that case their intersection is the entire common
plane, rather than a unique line.

## Python Animation

In [ ]:
import numpy as np
import plotly.graph_objects as go

# In Colab, if needed:
# !pip install plotly numpy

# ------------------------------------------------------------
# Geometry
# ------------------------------------------------------------
P = np.array([1.0, 0.0, 0.0])
Q = np.array([0.0, 1.0, 0.0])
O = np.zeros(3)

n = np.cross(P, Q)
n /= np.linalg.norm(n)


def R1(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [1, 0, 0],
        [0, c, -s],
        [0, s, c]
    ])


def R3(beta):
    c, s = np.cos(beta), np.sin(beta)
    return np.array([
        [c, -s, 0],
        [s, c, 0],
        [0, 0, 1]
    ])


def plane_trace(a, b, name, color, size=1.65):
    """Finite patch of the plane spanned by a and b."""
    corners = size * np.array([
        -a - b, a - b, a + b, -a + b
    ])

    return go.Mesh3d(
        x=corners[:, 0],
        y=corners[:, 1],
        z=corners[:, 2],
        i=[0, 0], j=[1, 2], k=[2, 3],
        color=color,
        opacity=0.28,
        flatshading=True,
        name=name,
        showlegend=True,
        hoverinfo="name"
    )


def line_trace(points, name, color, width=5, labels=None):
    points = np.asarray(points)

    return go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode="lines+markers+text" if labels else "lines",
        line=dict(color=color, width=width),
        marker=dict(color=color, size=3),
        text=labels,
        textposition="top center",
        textfont=dict(color=color, size=15),
        name=name,
        hoverinfo="name"
    )


def make_traces(theta_deg, beta_deg):
    theta, beta = np.deg2rad([theta_deg, beta_deg])
    R = R3(beta) @ R1(theta)

    PR, QR = R @ P, R @ Q
    nR = R @ n

    direction = np.cross(n, nR)
    magnitude = np.linalg.norm(direction)
    coincident = magnitude < 1e-10

    traces = [
        plane_trace(P, Q, "Plane OPQ", "#49a5ff"),
        plane_trace(PR, QR, "Plane OPᴿQᴿ", "#ff965e"),

        line_trace(
            [P, O, Q], "OP and OQ", "#49a5ff",
            labels=["P", "O", "Q"]
        ),
        line_trace(
            [PR, O, QR], "Rotated OP and OQ", "#ff965e",
            labels=["Pᴿ", "", "Qᴿ"]
        ),

        line_trace(
            [O, 1.8 * n], "Normal n", "#79c3ff",
            labels=["", "n"]
        ),
        line_trace(
            [O, 1.8 * nR], "Normal Rn", "#ffc397",
            labels=["", "Rn"]
        )
    ]

    # Keep the same number of traces for every slider position.
    if coincident:
        intersection_points = np.full((2, 3), np.nan)
    else:
        unit_direction = direction / magnitude
        intersection_points = np.array([
            -2.1 * unit_direction,
             2.1 * unit_direction
        ])

    traces.append(
        line_trace(
            intersection_points,
            "Intersection line",
            "#f6cc61",
            width=9
        )
    )

    return traces


# ------------------------------------------------------------
# Initial figure
# ------------------------------------------------------------
initial_theta = 55
initial_beta = 35

fig = go.Figure(
    data=make_traces(initial_theta, initial_beta)
)

axis_style = dict(
    range=[-2.2, 2.2],
    backgroundcolor="#101c2e",
    gridcolor="#27364c",
    zerolinecolor="#52617b"
)

fig.update_layout(
    title=dict(
        text="Two planes and their line of intersection",
        x=0.5
    ),
    template="plotly_dark",
    paper_bgcolor="#0b1220",
    font=dict(color="#e8eef7"),
    height=780,
    margin=dict(l=20, r=20, t=100, b=120),

    scene=dict(
        xaxis=dict(title="x₁", **axis_style),
        yaxis=dict(title="x₂", **axis_style),
        zaxis=dict(title="x₃", **axis_style),
        aspectmode="cube",
        camera=dict(eye=dict(x=1.5, y=1.7, z=1.25))
    ),

    legend=dict(x=0, y=1),
    uirevision="preserve-camera",

    annotations=[
        dict(
            x=0.5, y=1.08,
            xref="paper", yref="paper",
            text=(
                "n = x<sub>P</sub> × x<sub>Q</sub>"
                " &nbsp; | &nbsp; n<sub>R</sub> = Rn"
                " &nbsp; | &nbsp; "
                "Intersection: x = s(n × Rn)/‖n × Rn‖"
            ),
            showarrow=False,
            font=dict(size=14)
        ),
        dict(
            x=0.5, y=-0.17,
            xref="paper", yref="paper",
            text=(
                "Drag to rotate · Scroll to zoom"
                " · At θ = 0° or 180°, the planes coincide."
            ),
            showarrow=False,
            font=dict(size=13)
        )
    ]
)

# ------------------------------------------------------------
# Interactive controls
#
# JavaScript callbacks preserve both slider values independently.
# Plotly is embedded in the exported HTML: no internet required.
# ------------------------------------------------------------
html_controls = """
<div style="
    max-width:900px;margin:15px auto;padding:18px;
    background:#142033;color:#e8eef7;border-radius:12px;
    font-family:Arial,sans-serif;">
  <label>
    Tilt θ: <strong id="theta-value">55°</strong>
    <input id="theta-slider" type="range"
           min="0" max="180" step="1" value="55"
           style="width:100%;margin:12px 0;">
  </label>
  <label>
    Azimuth β: <strong id="beta-value">35°</strong>
    <input id="beta-slider" type="range"
           min="0" max="360" step="1" value="35"
           style="width:100%;margin:12px 0;">
  </label>
  <button id="animate-button">▶ Animate</button>
  <button id="coincide-button">Coincide</button>
  <button id="reset-button">Reset</button>
  <p id="geometry-status"></p>
</div>
"""

# Supply precomputed geometry for all integer slider positions
# indirectly through a compact JavaScript implementation.
post_script = """
const plot = document.getElementById('{plot_id}');
plot.insertAdjacentHTML('afterend', CONTROLS);

const thetaSlider = document.getElementById('theta-slider');
const betaSlider = document.getElementById('beta-slider');
const status = document.getElementById('geometry-status');
const animateButton = document.getElementById('animate-button');

let timer = null;

function updateGeometry() {
    const theta = Number(thetaSlider.value) * Math.PI / 180;
    const beta = Number(betaSlider.value) * Math.PI / 180;

    const c = Math.cos(beta), s = Math.sin(beta);
    const ct = Math.cos(theta), st = Math.sin(theta);

    const P = [c, s, 0];
    const Q = [-s * ct, c * ct, st];
    const nR = [s * st, -c * st, ct];

    // n = [0,0,1], so n × nR = [-nR_y,nR_x,0].
    const ell = [-nR[1], nR[0], 0];
    const ellNorm = Math.hypot(...ell);
    const same = ellNorm < 1e-10;

    const corners = [[-1,-1],[1,-1],[1,1],[-1,1]].map(
        ([u,v]) => P.map((p,k) => 1.65 * (u*p + v*Q[k]))
    );

    const segments = [
        corners,
        [P, [0,0,0], Q],
        [[0,0,0], nR.map(v => 1.8*v)],
        same
            ? [[null,null,null], [null,null,null]]
            : [-2.1,2.1].map(a => ell.map(v => a*v/ellNorm))
    ];

    Plotly.restyle(plot, {
        x: segments.map(points => points.map(p => p[0])),
        y: segments.map(points => points.map(p => p[1])),
        z: segments.map(points => points.map(p => p[2]))
    }, [1,3,5,6]);

    document.getElementById('theta-value').textContent =
        thetaSlider.value + '°';
    document.getElementById('beta-value').textContent =
        betaSlider.value + '°';

    const angle = Math.acos(
        Math.min(1, Math.abs(nR[2]))
    ) * 180 / Math.PI;

    status.textContent = same
        ? 'Coincident planes: their intersection is the entire plane.'
        : 'Distinct planes: the gold line is their intersection. '
          + 'Smaller plane angle: ' + angle.toFixed(1) + '°. '
          + '‖n × Rn‖ = ' + ellNorm.toFixed(3) + '.';
}

function stopAnimation() {
    clearInterval(timer);
    timer = null;
    animateButton.textContent = '▶ Animate';
}

thetaSlider.oninput = updateGeometry;
betaSlider.oninput = updateGeometry;

animateButton.onclick = () => {
    if (timer !== null) {
        stopAnimation();
        return;
    }
    animateButton.textContent = '❚❚ Pause';
    timer = setInterval(() => {
        betaSlider.value = (Number(betaSlider.value) + 2) % 360;
        updateGeometry();
    }, 100);
};

document.getElementById('coincide-button').onclick = () => {
    stopAnimation();
    thetaSlider.value = 0;
    updateGeometry();
};

document.getElementById('reset-button').onclick = () => {
    stopAnimation();
    thetaSlider.value = 55;
    betaSlider.value = 35;
    updateGeometry();
};

updateGeometry();
"""

import json
from pathlib import Path

post_script = post_script.replace(
    "CONTROLS", json.dumps(html_controls)
)

html = fig.to_html(
    full_html=True,
    include_plotlyjs=True,
    post_script=post_script,
    config=dict(
        responsive=True,
        displaylogo=False,
        toImageButtonOptions=dict(
            filename="plane_intersection",
            width=1400,
            height=1000
        )
    )
)

html = html.replace(
    "<body>",
    '<body style="margin:0;background:#0b1220;">'
)

output = Path("plane_intersection.html")
output.write_text(html, encoding="utf-8")
print(f"Saved: {output.resolve()}")

# Display the complete interactive visualization in Colab/Jupyter.
try:
    from IPython.display import HTML, display
    display(HTML(html))
except ImportError:
    import webbrowser
    webbrowser.open(output.resolve().as_uri())

# Q4

An inertial observer $\mathbf{e}$ notices that an object is moving in a straight line at a constant velocity. Show that all other inertial observers will also agree that the object is moving in a straight line at a constant velocity.

# Q5


An inertial observer $\mathbf{e}$ notices that an object is moving in a straight line at a constant acceleration (for example such as the free falling object under gravity). Show that other inertial observers need not agree that the object is moving in a straight line.

#Q6

An inertial observer  notices that an object is moving in a circle at a constant rate in the  frame with the center coniciding with the center of a the  frame. Show that other inertial observers need not agree that the object is moving in a circle.

# Answers to Q4, Q5 and Q6

Let $\mathbf{e}$ and $\mathbf{e}'$ represent two inertial observers with $\mathbf{e}'$ moving  parallel to $\mathbf{e}$ at a constant velocity $v=[v_1\:\:\:v_2\:\:\:v_3]^T$.

If $(t,x)$ is a representation of a point $P$ in $\mathbf{e}$ we have seen in Q5 that this same point has the representation $\left(t,(x-vt)\right)$ in the $\mathbf{e}'$ frame.

If the point $P$ is seen to be moving in a straight line by $\mathbf{e}$ (without loss of generality say a straight line through the origin) then $x(t)=f(t)\,a$ where $a=[a_1\:\:\:a_2\:\:\:a_3]$ is a constant and $f(\cdot)$ is some function of time. Then this point has the representation $\left(t,(f(t)\,a-vt)\right)$ in the $\mathbf{e}'$ frame.

## Answer to Q4

If $P$ is observed in $\mathbf{e}$ to move in a straight line at a constant velocity then $f(t)\simeq t$ and then in the $\mathbf{e}'$ frame this point has the representation $\left(t,t(a-v)\right)$ and hence is observed to be moving in a straight line as well.

In [ ]:
a=np.array([1,0,0]);
v=np.array([0,0,1])
tt=np.linspace(0,5,101)
zeta=np.zeros((len(tt),3));
for i,t in enumerate(tt):
  zeta[i,:]=t*(a-v);

In [ ]:
fig=mr.animate_particle_motion(zeta,[[-3,3],[-3,3],[-3,3]],'Motion in the e-prime frame')

## Answer to Q5

If $P$ is observed in $\mathbf{e}$ to move in a straight line at a constant acceleration then $f(t)\simeq t^2$ and then in the $\mathbf{e}'$ frame this point has the representation $\left(t,t(at-v)\right)$ and hence is observed to be moving in a **curved** path.

In [ ]:
a=np.array([0,0,-1]);
v=np.array([1,0,0])
tt=np.linspace(0,5,101)
zeta=np.zeros((len(tt),3));
for i,t in enumerate(tt):
  zeta[i,:]=t*(t*a-v);

In [ ]:
fig=mr.animate_particle_motion(zeta,[[-3,3],[-3,3],[-3,3]],'Motion in the e-prime frame')

## Answer to Q6

If $P$ is observed in $\mathbf{e}$ to move in a circle at a constant rate in the $\mathbf{e}_1,\mathbf{e}_2$ plane then $P$ has the representation $\left(t, (\sin{(\omega t)},\cos{(\omega t)},0)\right)$. Then in the $\mathbf{e}'$ frame that is moving in the $\mathbf{e}_3$ direction at a constant rate $v$ we see that this point has the representation $\left(t, (\sin{(\omega t)},\cos{(\omega t)},-vt)\right)$ and hence is observed to be moving in a **helical** path.

In [ ]:
v=np.array([0,0,1])
w=4;
tt=np.linspace(0,5,101)
zeta=np.zeros((len(tt),3));
for i,t in enumerate(tt):
    a=np.array([np.sin(w*t),np.cos(w*t),0]);
    zeta[i,:]=(a-v*t);

In [ ]:
fig=mr.animate_particle_motion(zeta,[[-3,3],[-3,3],[-3,3]],'Motion in the e-prime frame')

#Q7 - Rotating Cannon

n observer $\mathbf{e}'$ rides on a platform translating at constant velocity $v$ along the $\mathbf{e}_1$-axis relative to an inertial frame $\mathbf{e}$.
The two frames remain parallel to each other and $\mathbf{e}_3'$ is the vertically up direction. Every object experiences a cosntatant gravitational acceleration of magnitude $g$ acting vertically downwards as observed in $\mathbf{e}'$.

On the platform $\mathbf{e}'$, a cannon of muzzle length $l$ rotates in the $\mathbf{e}_3'\text{–}\mathbf{e}_1'$ plane, around the origin, at a constant counter clockwise rate of $\omega \, \mathrm{rad}/s$. The cannon fires a projectile with speed $u$, as measured in $\mathbf{e}'$.
At the instant of firing, the cannon points **vertically up** according to the observer $\mathbf{e}'$.

---

**Questions:**

1. Write down separately the description of the motion as observed in each frame.

2. Explain mathematically why the projectile appears to move **vertically** when observed by $\mathbf{e}'$, but follow a **parabolic** trajectory when observed in $\mathbf{e}$ if the cannon is not rotating but is still pointing vertically updaward at the time of firing.

3. Verify your conclusions using a simulation.
Plot or animate the trajectories in both frames for chosen parameters (e.g. $l=1\text{m},\, u = 20\text{m/s}, \, \omega = \frac{\pi}{4}\text{rad/s}, \, v = 5\text{m/s}, \, g = 9.8\text{m/s}^2$).


## Sample Answer

1. Write down separately the description of the motion as observed in each frame.

Assume without loss of generality that the two frames coincides at the time of the shooting of the cannon and that the both cloack are synchronized at that instant. Then the position of the cannon in the $\mathbf{e}'$ frame is $(t,x')$ where
\begin{align}
x'(t)&=\begin{bmatrix}
x_1'(t)\\ x_2'(t)\\ x_3'(t)
\end{bmatrix}
\end{align}

Since every object in $\mathbf{e}_3'$ feels a gravitational acceleration acting vertically downwards we have
\begin{align}
\ddot{x}'(t)&=\begin{bmatrix}
\ddot{x}_1'(t)\\ \ddot{x}_2'(t)\\ \ddot{x}_3'(t)
\end{bmatrix}=
-g\begin{bmatrix}
0\\ 0\\ 1
\end{bmatrix}
\end{align}
Thus by integrating twice we have
\begin{align}
{x}'(t)&=\begin{bmatrix}
{x}_1'(t)\\ {x}_2'(t)\\ {x}_3'(t)
\end{bmatrix}=\begin{bmatrix}
0 \\ 0\\ l
\end{bmatrix}+
\begin{bmatrix}
-l\omega t\\0 \\ ut
\end{bmatrix}
-\frac{gt^2}{2}\begin{bmatrix}
0\\ 0\\ 1
\end{bmatrix}=\begin{bmatrix}
-l\omega t\\ 0 \\ l+ut-\frac{gt^2}{2}
\end{bmatrix}
\end{align}
From the Galilean transformations we have that the postion of the cannon in the $\mathbf{e}$ frame is described by
\begin{align}
{x}(t)&=x'(t)+\begin{bmatrix}v t\\0\\0
\end{bmatrix}=
\begin{bmatrix}
(v-l\omega) t\\ 0 \\ l+ut-\frac{gt^2}{2}
\end{bmatrix}
\end{align}

---

2. When $\omega=0$
\begin{align}
{x}'(t)&=\begin{bmatrix}
0 \\ 0 \\ l+ut-\frac{gt^2}{2}
\end{bmatrix}.
\end{align}
Thus the motion appears vertical in the $\mathbf{e}'$ frame.

In the $\mathbf{e}$ frame is described by
\begin{align}
{x}(t)&=x'(t)+\begin{bmatrix}v t\\0\\0
\end{bmatrix}=
\begin{bmatrix}
v t\\ 0 \\ l+ut-\frac{gt^2}{2}
\end{bmatrix}
\end{align}
and hence we have
\begin{align}
x_3&=l+\frac{u}{v}x_1-\frac{g}{2v^2}x_1^2
\end{align}
and hence the motion is parabolic in $\mathbf{e}$

---

3. Simulation verification.
Plot or animate the trajectories in both frames for chosen parameters (e.g. $l=1 m, \,u = 20\text{m/s}, , \omega = \pi/4\,\text{rad/s}, \, v = 5\text{m/s}, \, g = 9.8\text{m/s}^2$).



In [ ]:
import numpy as np
import plotly.graph_objects as go

# Your pre-defined animation method (must exist in the same scope)
# def animate_particle_motion(self, xx, axis_range, fig_title): ...

# --- Simulation setup ---

# Parameters
l = 1.0        # m
omega = 0 #np.pi/4  # rad/s
u = 20     # m/s
v = 5 #l*omega        # m/s
g = 9.8        # m/s^2
t_max = 2.5    # seconds of flight
dt = 0.02
t = np.arange(0, t_max, dt)

# --- Trajectory in e' frame (rotating cannon on moving platform) ---
# Using your derived expressions:
# x'(t) = [ l*omega*t, 0, l + u*t - (1/2)*g*t^2 ]

x1p = -l * omega * t
x2p = np.zeros_like(t)
x3p = l + u * t - 0.5 * g * t**2

# Stack into a list of tuples for the function call
xx_prime = list(zip(x1p, x2p, x3p))

# --- Trajectory in e frame ---
# x(t) = x'(t) + [v*t, 0, 0]
x1 = (v - l * omega) * t
x2 = np.zeros_like(t)
x3 = l + u * t - 0.5 * g * t**2

xx = list(zip(x1, x2, x3))

# --- Axis ranges (same for both plots) ---
axis_range = [(-1, 20), (-1, 1), (0, 30)]

# --- Create two separate animations ---

# Assuming you're calling this from a class instance, say 'simulator'
# Replace `self` by the instance name if applicable, e.g. simulator.animate_particle_motion(...)
# If not inside a class, you can temporarily adapt it like:


# Animation in e' frame
fig_prime = mr.animate_particle_motion(
    xx=xx_prime,
    axis_range=axis_range,
    fig_title="Projectile Motion in Moving Frame e′ (\\omega=\\pi/4 rad/s)"
)

# Animation in e frame
fig = mr.animate_particle_motion(
    xx=xx,
    axis_range=axis_range,
    fig_title="Projectile Motion in Ground Frame e"
)


# Q8 **Answers to Michelson-Morley Experiment**


**Light beam along the longitudinal arm (along $\mathbf{e}_1'$):**

According to the Galilean transformation, the light beam traveling **forward** along $\mathbf{e}_1'$ has speed $c - v$ relative to the apparatus, and the beam traveling **backward** after reflection has speed $c + v$.

The total travel time along the longitudinal arm is
\begin{align}
T_{\parallel} &= \frac{L}{c - v} + \frac{L}{c + v} = \frac{2Lc}{c^2 - v^2}
= \frac{2L}{c}\left(1 + \frac{v^2}{c^2} + \dots \right).
\end{align}

**Light beam along the transverse arm (along $\mathbf{e}_2'$):**

For the beam traveling along the transverse arm, the apparatus moves with velocity $v$ during the beam's transit.
To reach the mirror, the light must follow a diagonal path such that its velocity relative to $\mathbf{e}$ satisfies
\begin{align}
c^2 &= v^2 + u_\perp^2.
\end{align}
Thus the time for one leg of the transverse path is $\frac{L}{\sqrt{c^2 - v^2}}$, so that the total round-trip time is
\begin{align}
T_{\perp} = \frac{2L}{\sqrt{c^2 - v^2}}
&= \frac{2L}{c}\left(1 + \frac{v^2}{2c^2} + \dots \right).
\end{align}

The time difference between the returned litght beams is
\begin{align}
\Delta T &= T_{\parallel} - T_{\perp},
\end{align}
and to leading order in $v^2/c^2$,
\begin{align}
\Delta T \approx \frac{L v^2}{c^3}.
\end{align}

---

**Expected fringe shift:**



When the interferometer is rotated by $90^\circ$, the roles of the arms are interchanged the change in the phase difference between the two instances of the beam is

The phase difference between the two beams for the first orientation of the interferometer:
\begin{align}
\Delta T_1 &= \frac{L v^2}{c^3}
\quad \Rightarrow \quad
\Delta \phi_1 = \frac{2\pi c}{\lambda} \Delta T_1
= \frac{2\pi L v^2}{\lambda c^2}.
\end{align}

After rotating the apparatus by $90^\circ$, the roles of the arms are interchanged:
\begin{align}
\Delta T_2 = -\frac{L v^2}{c^3}
\quad \Rightarrow \quad
\Delta \phi_2 = -\frac{2\pi L v^2}{\lambda c^2}.
\end{align}

Hence, the **total change in phase difference** due to the rotation is
\begin{align}
\Delta \phi_{\text{change}} &= \Delta \phi_2 - \Delta \phi_1
= -2\Delta \phi_1
= -\frac{4\pi L v^2}{\lambda c^2}.
\end{align}

The corresponding **number of fringes shifted** is
\begin{align}
N &= \frac{|\Delta \phi_{\text{change}}|}{2\pi}
= \frac{2L v^2}{\lambda c^2}.
\end{align}

---

**Discussion:**

Here are several **resources** with excellent explanations, diagrams, and even interactive simulations of the **Michelson-Morley experiment**.


1. **HyperPhysics — Michelson–Morley Experiment (Georgia State University)**
   🔗 [http://hyperphysics.phy-astr.gsu.edu/hbase/phyopt/michel.html](http://hyperphysics.phy-astr.gsu.edu/hbase/phyopt/michel.html)

   > Clear conceptual explanation with schematic diagrams and key equations.
   > Shows both Galilean and relativistic interpretations.

2. **University of Virginia - Galileo and Einstein: The Michelson-Morley Experiment**
    > This resource offers a clear explanation of the historical context and the "ether wind" hypothesis it aimed to test. It features a great interactive Michelson-Morley Experiment applet that helps visualize the experiment's setup and the expected results if an aether existed.
    [URL](https://galileo.phys.virginia.edu/classes/252/michelson.html#michmor)


3. **Wikipedia — Michelson–Morley Experiment**
   🔗 [https://en.wikipedia.org/wiki/Michelson-Morley_experiment](https://en.wikipedia.org/wiki/Michelson%E2%80%93Morley_experiment)

   > Comprehensive overview with diagrams of the interferometer setup, experimental details, and historical background.






